In [ ]:
from pathlib import Path
import os
from os import PathLike
from typing import Optional, Sequence, Union
from fastcore.basics import patch

from trouver.obsidian.vault import NoteDoesNotExistError, VaultNote, note_name_from_path, all_paths_to_notes_in_vault, NoteNotFoundInCacheError, NoteNotUniqueError, NotePathIsNotIdentifiedError
from trouver.obsidian.links import ObsidianLink, LinkType, replace_links_in_text
from trouver.obsidian.vault import path_to_obs_id

In [ ]:
import shutil
import tempfile
from unittest import mock

from fastcore.test import *
from nbdev.showdoc import show_doc

from trouver.helper.tests import _test_directory

In [ ]:
#| export obsidian.vault
@patch(cls_method=True)
def update_cache(
        cls: VaultNote,
        vault: PathLike # The vault.
    ) -> None:
    r"""Class method to update cache for `vault` by inspecting all files
    in subdirectories of `vault`."""
    cls.cache[str(vault)] = all_paths_to_notes_in_vault(
        vault, as_dict=True)

        

In [ ]:
#| export obsidian.vault
@patch(cls_method=True)
def clear_cache(cls: VaultNote):
    r"""Class method to clear out the entire cache for all vaults."""
    cls.cache = {}
        

In [ ]:
#| export obsidian.vault
@patch(cls_method=True)
def _add_single_entry_to_cache(
        cls: VaultNote,
        vault: PathLike,
        rel_path: PathLike) -> None:
    r"""Adds a single entry for a note to the cache.
    
    Does nothing if the entry already exists.

    **Parameters**
    - vault - PathLike
    - rel_path - PathLike
        - The path to the note, relative to `vault`.
    """
    vault_str = str(vault)
    if vault_str not in cls.cache:
        cls.cache[vault_str] = {}
    name = note_name_from_path(rel_path)
    if name not in cls.cache[vault_str]:
        cls.cache[vault_str][name] = []
    if rel_path not in cls.cache[vault_str][name]:
        cls.cache[vault_str][name].append(str(rel_path))

@patch(cls_method=True)
def _remove_single_entry_from_cache(
        cls: VaultNote,
        vault: PathLike,
        rel_path: PathLike) -> None:
    r"""Removes a single entry for a note from the cache.

    Does nothing if the entry is not already there.

    **Parameters**
    - vault - PathLike
    - rel_path - PathLike
        - The path to the note, relative to `vault`.
    """
    vault_str = str(vault)
    if vault_str not in cls.cache:
        return
    name = note_name_from_path(rel_path)
    if name not in cls.cache[vault_str]:
        return
    rel_path = Path(rel_path)
    cls.cache[vault_str][name] = [
        cache_path for cache_path in cls.cache[vault_str][name]
        if Path(cache_path) != rel_path]

#### Cache of the `VaultNote` class

In [ ]:
show_doc(VaultNote.update_cache)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L665){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.update_cache

>      VaultNote.update_cache (vault:os.PathLike)

*Class method to update cache for `vault` by inspecting all files
in subdirectories of `vault`.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| vault | PathLike | The vault. |
| **Returns** | **None** |  |

In [ ]:
show_doc(VaultNote.clear_cache)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L678){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.clear_cache

>      VaultNote.clear_cache ()

*Class method to clear out the entire cache for all vaults.*

The `VaultNote` class keeps a cache of the notes (files with extension `.md`) in the vault. In pracctice, this cache is updated when a note of the specified name is not found when constructing a `VaultNote` instance via the `name` parameter.

In [ ]:
#| export obsidian.vault
@patch(cls_method=True)
def _check_name_exists_and_unique_in_vault_cache(
        cls: VaultNote,
        vault: PathLike, # The vault
        name: str # The name
        ) -> None:
    r"""
    Raise `NoteNotUniqueError` or `NoteNotFoundInCacheError` if the note of
    the specified name is not unique or is not found in the cache.

    Note that a note may exist but not be found in the cache if it was
    created without using the `VaultNote.create` method. For example,
    this could happen if a note is created by the user using the
    file explorer.
    
    **Parameters**
    - vault - PathLike
    - name - str

    **Raises**
    - `NoteNotUniqueError`
    - `NoteNotFoundInCacheError`.
    """
    vault_dict = cls.cache[str(vault)]
    if not name in vault_dict or len(vault_dict[name]) == 0:
        raise NoteNotFoundInCacheError.from_note_name(name)
    elif len(vault_dict[name]) > 1:
        raise NoteNotUniqueError.from_note_names(
            name, vault_dict[name])
    else: #name in vault_dict and len(vault_dict[name]) == 1
        note_path = vault_dict[name][0]
        if not os.path.exists(Path(vault) / note_path):
            raise NoteDoesNotExistError.from_note_name(name)

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    VaultNote.update_cache(temp_vault)
    
    name_of_unique_note = 'exponential_function'
    assert VaultNote._check_name_exists_and_unique_in_vault_cache(temp_vault, name_of_unique_note) is None

    with ExceptionExpected(ex=NoteNotFoundInCacheError):
        non_existent_note_name = 'this_note_does_not_exist'
        VaultNote._check_name_exists_and_unique_in_vault_cache(temp_vault, non_existent_note_name)

    with ExceptionExpected(ex=NoteNotFoundInCacheError):
        # The following note file is created without using the `VaultNote.cretae` method.
        # As such, the cache is not updated with this new note.
        to_create_note_name = 'note_created_without_using_VaultNote_create'
        open(temp_vault / f'{to_create_note_name}.md', 'w').close()
        VaultNote._check_name_exists_and_unique_in_vault_cache(temp_vault, to_create_note_name)

    with ExceptionExpected(ex=NoteDoesNotExistError):
        # The following note file is deleted (without updating the cache).
        # As such, while the cache indicates that
        # the file may exist, The `VaultNote._check_name_exists_and_unique_in_vault_cache`
        # actually checks whether the file exists and concludes that it does not.
        to_delete_note_name = 'note_1'
        to_delete_note_rel_path = f'topology/{to_delete_note_name}.md'
        os.remove(temp_vault / to_delete_note_rel_path)
        VaultNote._check_name_exists_and_unique_in_vault_cache(temp_vault, to_delete_note_name)

    with ExceptionExpected(ex=NoteNotUniqueError):
        existent_but_non_unique_note_name = 'ring'
        VaultNote._check_name_exists_and_unique_in_vault_cache(temp_vault, existent_but_non_unique_note_name)

    with ExceptionExpected(ex=NoteNotUniqueError):
        # In this example, the note name is not unique. 
        # The note file is also deleted without updating the cache.
        # Nevertheless, the `VaultNote._check_name_exists_and_unique_in_vault_cache`
        # determines that the note name is not unique.
        to_delete_note_name = 'category'
        to_delete_note_rel_path = f'category_theory/{to_delete_note_name}.md'
        os.remove(temp_vault / to_delete_note_rel_path)
        VaultNote._check_name_exists_and_unique_in_vault_cache(temp_vault, to_delete_note_name)

In [ ]:
VaultNote.clear_cache()
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    assert VaultNote._check_if_cache_needs_to_update(temp_vault, name='ring')
    VaultNote.update_cache(temp_vault)
    assert VaultNote._check_if_cache_needs_to_update(temp_vault, name='does_not_exist')

In [ ]:
# repr(VaultNote(vault='.', name='hi'))

In [ ]:
#| export obsidian.vault
@patch
def _identify_rel_path(
        self: VaultNote
        ) -> Union[Path, None]:
    r"""Returns the Path to the note that this object represents.

    More precisely, if `rel_path` is specified at construction or
    if `self.rel_path` is already identified, then this method
    returns that path. Otherwise, this method looks into the cache to
    check if a note of the `name` specified at construction is in the
    vault and returns the first note in the list of the `name` in the
    cache. If no such note exists, then this method reutrns `None`.
    """
    if self.rel_path is not None:
        return self.rel_path
    cache_search = self.__class__._get_from_cache(self.vault, self.name)
    if cache_search:  # `cache_search` could be `None` or an empty list.
        return cache_search[0]
    return None

@patch
def identify_rel_path(
        self: VaultNote,
        update_cache=False # If `True`, if the cache is searched, and if a note of the specified name is not found in the cache, then the cache is updated and searched again. Defaults to `False`.
        ) -> None:
    r"""Sets `self.rel_path` to a path, if not already done so.

    If `self.rel_path` is not already set as a path, then the cache
    is searched to find a note whose name is `self.name` (which is
    necessarily specified).
    """
    rel_path = self._identify_rel_path()
    if rel_path is not None:
        self.rel_path = rel_path
    elif update_cache:
        self.__class__.update_cache(self.vault)
        self.identify_rel_path(update_cache=False)

In [ ]:
#| export obsidian.vault
@patch
def rel_path_identified(self: VaultNote) -> bool:
    r"""Return `True` if `self.rel_path` is identified, i.e. is a path
    that is not `None`."""
    return self.rel_path is not None